# Project 1 (open models): An LLM as a coder for qualitative interviews

**Hands-on project, about 2 hours, in teams or alone.**

In December 2025 Anthropic released **1,250 interview transcripts** in which professionals describe how they use AI at work and how they feel about it. The interviews were run by an AI interviewer, and every participant agreed to public release. The data sit on Hugging Face as [`Anthropic/AnthropicInterviewer`](https://huggingface.co/datasets/Anthropic/AnthropicInterviewer) and come in three groups:

| Group | Interviews |
|---|---|
| `workforce` (general workforce) | 1,000 |
| `creatives` (writers, artists, musicians, designers) | 125 |
| `scientists` | 125 |

**Main research questions:** *Do the three occupational groups differ in their attitude towards AI and in what worries them most?*

A qualitative researcher would answer this by reading every transcript and coding it with a codebook. 

**You do not write any Python.** Run the cells top to bottom. Wherever a cell says `TODO`, you make a research decision: write a prompt in plain English, choose a setting, or pick a model. Replace the `...` with your choice and run the cell. If you forget one, the cell stops and tells you which value is missing.


> **Open-model version.** This notebook runs an open-weight model (**Qwen2.5-3B-Instruct** by default) on your own GPU through Hugging Face `transformers`, instead of calling the OpenAI API. No API key is needed, and no text leaves your machine, which matters for confidential material such as interviews. Only the setup cells differ from the API version. Everything after them is identical, so you can compare the two models' codes directly. A GPU is needed; a free Colab or Kaggle T4 is enough.

## Setup

Run these cells first. The first installs the libraries (needed on Colab or Kaggle). The second loads an open-weight model onto the GPU. The third defines three helpers you do not need to edit:

- `ask()` sends a prompt to the local model and returns the reply. With `json_mode=True` it **prefills** the answer with `{`, so the model has to continue a JSON object instead of starting with "Sure! Here is...". This is the technique from the *Prompting* demo. Its `model=` argument sends the prompt to a different loaded model instead.
- `parse_json()` turns that reply into a Python dict, or `{}` if it is not valid JSON.
- `run_parallel()` has the same name as in the API version so the later cells run unchanged. A single local model answers one request at a time, so it just loops with a progress bar.
- `require()` stops a cell with a clear message if you have not yet filled in one of its `TODO` values.

**Which model?** `Qwen2.5-3B-Instruct` (about 6 GB) fits a T4 and is the smallest Qwen model that follows a codebook reasonably well. `Qwen2.5-1.5B-Instruct` is faster but noticeably worse at following instructions. With 24 GB or more of GPU memory, `Qwen2.5-7B-Instruct` is also an option. Try them all. 

In [ ]:
!pip install -q datasets

In [ ]:
import re
import json
import warnings

import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from transformers import pipeline
from transformers import logging as hf_logging

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()

MODEL = "Qwen/Qwen2.5-3B-Instruct"

pipe = pipeline(
    "text-generation",
    model=MODEL,
    dtype="auto",          # on a Colab T4 use dtype=torch.float16: bfloat16 is emulated there and slow
    device_map="auto",
)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Model : {MODEL} on {pipe.model.device} ({pipe.model.dtype})")

In [ ]:
def ask(system, user, json_mode=False, model=None, **decoding):
    """Send one system + user message pair to a local model and return the reply text.

    decoding: generation parameters passed straight to the pipeline, e.g.
              do_sample=False, max_new_tokens=200, temperature=0.7, top_p=0.9.
    model: another loaded pipeline to use instead of `pipe`, e.g. a second coder.
    """
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    if json_mode:
        # Prefill: the model continues this assistant message, so its reply starts inside the JSON.
        messages.append({"role": "assistant", "content": "{"})

    generator = pipe if model is None else model
    reply = generator(messages, **decoding)[0]["generated_text"][-1]["content"]
    if json_mode and not reply.lstrip().startswith("{"):
        reply = "{" + reply          # some versions return only the continuation
    return reply.strip()


def parse_json(reply):
    """Turn the model's reply into a dict; return {} if it is not valid JSON."""
    start = reply.find("{")
    if start == -1:
        return {}
    try:
        parsed, _ = json.JSONDecoder().raw_decode(reply[start:])   # ignores anything after the closing brace
        return parsed if isinstance(parsed, dict) else {}
    except json.JSONDecodeError:
        return {}


def run_parallel(fn, items, workers=None):
    """Apply fn to every item, in order. Same name as in the API version; a local model runs one at a time."""
    return [fn(item) for item in tqdm(items)]


def require(**values):
    """Stop with a clear message if a TODO value is still ... ."""
    missing = [name for name, value in values.items() if value is ... or value == "..."]
    if missing:
        raise ValueError(f"Fill in the TODO first: {', '.join(missing)}")

## Step 1 - Load the data and read before you code

Each transcript alternates between the interviewer (`AI:` or `Assistant:`) and the participant (`User:`). The interviewer often summarises what the participant said, sometimes more warmly than the participant did. We therefore code **only the participant's answers**, which `participant_text()` extracts.

You choose how many interviews per group to code. We sample the same number from each group because the groups are very unequal in size: a random 60 from all 1,250 would contain only about 6 creatives and 6 scientists. More interviews give more precise proportions but take longer to code. On a T4, allow roughly 5 seconds per interview per model.

In [5]:
TURN = re.compile(r"^(User|AI|Assistant):", re.MULTILINE)


def participant_text(transcript):
    """Keep only what the interviewee said, one answer per paragraph."""
    parts = TURN.split(transcript)
    # parts looks like ['', 'Assistant', ' Hi! ...', 'User', ' Sounds good ...', 'AI', ...]
    answers = [text.strip() for speaker, text in zip(parts[1::2], parts[2::2]) if speaker == "User"]
    return "\n\n".join(answers)


ds = load_dataset("Anthropic/AnthropicInterviewer")
interviews = pd.concat(
    [ds[split].to_pandas().assign(group=split) for split in ds],
    ignore_index=True,
)
interviews["answers"] = interviews["text"].apply(participant_text)

# TODO: how many interviews per group? 10 = quick test, 20 = recommended, 30 = slower but more precise.
N_PER_GROUP = 5
require(N_PER_GROUP=N_PER_GROUP)
sample = (
    interviews.groupby("group")
    .sample(n=N_PER_GROUP, random_state=42)
    .reset_index(drop=True)
)

print(interviews["group"].value_counts(), "\n")
print(f"Sample: {len(sample)} interviews")
print(f"Answer length in the sample: median {sample['answers'].str.len().median():.0f} characters")

Now **read**. The next cell prints one interview from each group. Each team member reads one of them and notes, in a few words: *How does this person feel about AI in their work? What worries them most?*

Do this before looking at the codebook in Step 2. The themes you notice yourselves are the ones you will check the codebook against.

In [6]:
# TODO: each team member reads one of the interviews below and writes down, in a few words,
#       how the person feels about AI in their work and what worries them most.
for i in [0, N_PER_GROUP, 2 * N_PER_GROUP]:     # one interview per group: creatives, scientists, workforce
    row = sample.iloc[i]
    print("=" * 80)
    print(f"{row.transcript_id}  |  group: {row.group}\n")
    print(row.answers, "\n")

## Step 2 - Adapt the codebook

The codebook below codes two variables per interview:

- `attitude`: the overall stance towards AI in the participant's own work.
- `main_concern`: the **one** concern the participant stresses most.

It is a starting point, not a finished instrument. Compare it with the notes you made while reading. If a definition does not fit what you read, you can change its text (between the quotes) and run the cell again. Otherwise just run it.

Every change is a methodological decision. Keep a short list of what you changed and why; you will need it in Step 7.

In [ ]:
ATTITUDE = {
    "positive": "sees AI mainly as helpful in their work; few or no reservations",
    "mixed":    "describes clear benefits AND clear reservations",
    "negative": "mainly sceptical, reluctant or worried about AI in their work",
}

CONCERN = {
    "job_security":     "fear of losing work, income or clients to AI; devaluation of the profession",
    "skill_loss":       "worry about losing their own skills, learning less, or becoming dependent on AI",
    "accuracy":         "errors, hallucinations, unreliable output, having to check everything",
    "authenticity":     "loss of personal voice, creativity, originality or the human touch",
    "ethics_ip":        "copyright, training data, plagiarism, fairness to other creators",
    "privacy":          "confidential or client data, security",
    "social_judgement": "stigma; colleagues or clients disapproving of AI use; hiding their use",
    "none":             "the participant expresses no real concern",
}

# TODO (optional): if a definition does not fit what you read, change its text (keep the quotes)
#       and write down what you changed and why.

## Step 3 - Write the prompt and code one interview

For one category per response, a single word was enough in the survey exercise. Here we want three things back from every interview: two codes and a **supporting quote**. The quote is there for you, not for the model. It makes every code checkable: if the model cannot point to where the participant said it, the code is suspect. Models do sometimes invent quotes, so `quote_found()` checks whether the quote really appears in the answers.

`build_prompt()` at the top of the next cell already puts the codebook and the interview into the prompt. **You write the last part: the instruction that tells the model what to return**, which is appended at its end. Write it in plain English. A good instruction:

- asks for a **JSON object** with exactly three keys: `attitude`, `main_concern` and `quote`;
- names the **allowed labels** for each key (copy them from Step 2) and says to use them exactly as written;
- says what the quote must be: copied **word for word** from the answers, short (e.g. at most 25 words), and what to do when there is no concern;
- asks for the JSON object **only**, with nothing before or after it.

Then choose the **decoding settings** (see the *Basics of Text Generation* demo):

- `DO_SAMPLE`: `False` means greedy decoding, so the same interview always gets the same code. `True` samples, so codes can change from run to run. Which one do you want for a coding study?
- `TEMPERATURE`, `TOP_P`, `TOP_K`: only used when `DO_SAMPLE = True` (set them to `None` otherwise). They control how adventurous the sampling is.
- `REPETITION_PENALTY`: `1.0` switches it off. Careful: the penalty also applies to words that appear in the **prompt**, so a high value pushes the model away from the participant's own words, which is exactly what a verbatim quote needs.
- `MAX_NEW_TOKENS`: the room the model has for its reply. The JSON with a 25-word quote needs roughly 80-150 tokens. Too little, and the JSON is cut off and cannot be read (it shows up as a missing code).

In [ ]:
def describe(scheme):
    return "\n".join(f"- {label}: {definition}" for label, definition in scheme.items())

# TODO: write the system message that tells the model what it is and what it should do. It should be a few sentences, in plain English.
SYSTEM = ...


def build_prompt(answers):
    """The user message sent for one interview. Your OUTPUT_INSTRUCTION goes at the end."""
    return (
        "Below are one participant's answers from an interview about using AI at work.\n\n"
        f"Variable 1 - attitude towards AI in their own work:\n{describe(ATTITUDE)}\n\n"
        f"Variable 2 - main_concern, the single concern the participant stresses most:\n{describe(CONCERN)}\n\n"
        f'Interview answers:\n"""\n{answers}\n"""\n\n'
        + OUTPUT_INSTRUCTION.strip()
    )


# TODO: write the instruction that tells the model what to return (plain English, between the triple quotes).
OUTPUT_INSTRUCTION = """
...
"""

# TODO: choose the decoding settings.
DO_SAMPLE = ...            
TEMPERATURE = ...          
TOP_P = ...                
TOP_K = ...                
REPETITION_PENALTY = ...   
MAX_NEW_TOKENS = ...       

In [ ]:
require(OUTPUT_INSTRUCTION=OUTPUT_INSTRUCTION.strip(), SYSTEM=SYSTEM, DO_SAMPLE=DO_SAMPLE,
        TEMPERATURE=TEMPERATURE, TOP_P=TOP_P, TOP_K=TOP_K,
        REPETITION_PENALTY=REPETITION_PENALTY, MAX_NEW_TOKENS=MAX_NEW_TOKENS)

DECODING = dict(do_sample=DO_SAMPLE, repetition_penalty=REPETITION_PENALTY, max_new_tokens=MAX_NEW_TOKENS)
if DO_SAMPLE:
    DECODING.update(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K)
print("Decoding:", DECODING)


def code_interview(answers, model=None):
    return parse_json(ask(SYSTEM, build_prompt(answers), json_mode=True, model=model, **DECODING))


def quote_found(quote, answers):
    """True if the quote really appears in the answers (ignoring case, spacing and quote marks)."""
    def norm(s):
        s = str(s).replace("’", "'").replace("“", '"').replace("”", '"')
        return re.sub(r"\s+", " ", s).lower().strip(" .\"'")
    return norm(quote) in norm(answers)


result = code_interview(sample.loc[0, "answers"])
print(json.dumps(result, indent=2, ensure_ascii=False))
print("\nQuote found in the answers:", quote_found(result.get("quote", ""), sample.loc[0, "answers"]))

Compare the result with your own reading of interview 0. Would you have coded it the same way? If the quote is not found, look at it: is it a paraphrase, a quote stitched together from two places, or an invention? If the reply is empty or a label is wrong, improve `OUTPUT_INSTRUCTION` or the decoding settings (e.g. `MAX_NEW_TOKENS`) and run the two cells again.

## Step 4 - Code the whole sample

This sends one request per interview to the local model. The results are saved to `p1_llm_codes.csv`, so a crashed kernel does not cost you the run.

The checks run automatically. They catch the three silent failures of LLM coding: a label outside the codebook, a reply that was not valid JSON (it shows up as a missing label), and an invented quote.

In [ ]:
results = run_parallel(code_interview, sample["answers"].tolist())

coded = sample.join(pd.DataFrame(results, columns=["attitude", "main_concern", "quote"]))
coded["quote_ok"] = [quote_found(q, a) for q, a in zip(coded["quote"], coded["answers"])]
coded.drop(columns=["text"]).to_csv("p1_llm_codes.csv", index=False)

print(f"Attitude outside the codebook : {(~coded['attitude'].isin(ATTITUDE)).sum()} of {len(coded)}")
print(f"Concern outside the codebook  : {(~coded['main_concern'].isin(CONCERN)).sum()} of {len(coded)}")
print(f"Quotes not found in the text  : {(~coded['quote_ok']).sum()} of {len(coded)}")
print()
print(coded["main_concern"].value_counts())

In [ ]:
# TODO: read the interviews that failed. Is there a pattern? If so, improve OUTPUT_INSTRUCTION,
#       the codebook or the decoding settings, and run Steps 3 and 4 again.
failed = coded[
    ~coded["attitude"].isin(ATTITUDE)
    | ~coded["main_concern"].isin(CONCERN)
    | ~coded["quote_ok"]
]

print(f"{len(failed)} interviews failed a check\n")
for row in failed.itertuples():
    print(f"{row.transcript_id} ({row.group})  attitude={row.attitude}  main_concern={row.main_concern}")
    print(f"   quote: {row.quote!r}\n")

## Step 5 - A second coder from a different model family

Everything so far says how *one* model coded, not whether its codes can be trusted. In a coding study a second coder codes the same material independently, and you report how far the two agree. There is no time to code the interviews by hand today, so the second coder is another LLM.

It should come from a **different model family**. Models of one family (Qwen2.5-1.5B, -3B, -7B, ...) are trained on largely the same data with the same recipe, so they tend to make the same mistakes. Two of them agreeing is like two research assistants trained by the same supervisor agreeing: reassuring, but not independent. Qwen is already taken, so pick something else.

**Find the model on [Hugging Face](https://huggingface.co/models)** (about 10 minutes):

1. Filter by the task **Text Generation** and search for `instruct`. Instruction-tuned models follow a codebook; base models only continue text.
2. Keep it small: **at most about 3B parameters** (shown on the model page, often in the name). Both models have to fit on the GPU together. A T4 has 15 GB, and the first model already uses about 6 GB.
3. Avoid **gated** models: a padlock and "You need to agree to share your contact information to access this model". They need an account, an approved licence request and a login token, which takes longer than you have. Several well-known families, including Llama and Gemma, are gated.
4. Read the model card: the licence, the languages it was trained on, and the context length. The answers run to about 3,000 tokens, so the context has to be at least 4k.
5. Avoid *reasoning* models that write a long `<think>...</think>` section before answering, unless the card shows how to switch that off: the prefilled `{` would be ignored.

Families worth a look: Phi (Microsoft), Granite (IBM), SmolLM (Hugging Face), OLMo (Allen AI), Mistral.

Paste its id (the `organisation/model-name` part of the URL) into `SECOND_MODEL` and run the cell. It codes all 60 interviews with the **same codebook and prompt**; only the model changes.

In [ ]:
# TODO: paste the id of the model you found on Hugging Face between the quotes, e.g. "organisation/model-name".
SECOND_MODEL = "..."

require(SECOND_MODEL=SECOND_MODEL)

pipe2 = pipeline(
    "text-generation",
    model=SECOND_MODEL,
    dtype="auto",          # on a Colab T4 use dtype=torch.float16
    device_map="auto",
)
print(f"Second coder: {SECOND_MODEL} on {pipe2.model.device} ({pipe2.model.dtype})")

second = run_parallel(lambda answers: code_interview(answers, model=pipe2), coded["answers"].tolist())
second = pd.DataFrame(second, columns=["attitude", "main_concern", "quote"], index=coded.index)

coded["attitude_2"] = second["attitude"]
coded["main_concern_2"] = second["main_concern"]
coded["quote_2"] = second["quote"]
coded["quote_ok_2"] = [quote_found(q, a) for q, a in zip(coded["quote_2"], coded["answers"])]
coded.drop(columns=["text"]).to_csv("p1_llm_codes.csv", index=False)

print(f"Attitude outside the codebook : {(~coded['attitude_2'].isin(ATTITUDE)).sum()} of {len(coded)}")
print(f"Concern outside the codebook  : {(~coded['main_concern_2'].isin(CONCERN)).sum()} of {len(coded)}")
print(f"Quotes not found in the text  : {(~coded['quote_ok_2']).sum()} of {len(coded)}")

If the checks above show many labels outside the codebook or many missing values, the second model is struggling with the format rather than the content. If that happens, pick another model and run the cell again.

The next cell compares the two coders. It reports two numbers per variable: the percentage agreement and **Cohen's kappa**. Kappa corrects for the agreement you would get by chance. That matters here: if most participants are `mixed`, two coders who always write `mixed` already agree most of the time.

In [ ]:
# TODO: run the cell. Do the two models agree more on attitude or on main_concern? Is kappa much lower
#       than the raw agreement, and why?
from sklearn.metrics import cohen_kappa_score

for first, second_col in [("attitude", "attitude_2"), ("main_concern", "main_concern_2")]:
    agreement = (coded[first] == coded[second_col]).mean()
    kappa = cohen_kappa_score(coded[first].astype(str), coded[second_col].astype(str))   # a missing label counts as a disagreement
    print(f"{first:13s} agreement: {agreement:.0%}   kappa: {kappa:.2f}")

disagree = coded[(coded["attitude"] != coded["attitude_2"]) | (coded["main_concern"] != coded["main_concern_2"])]
print(f"\nDISAGREEMENTS: {len(disagree)} of {len(coded)} interviews\n")
print(disagree[["transcript_id", "group", "attitude", "attitude_2", "main_concern", "main_concern_2"]].to_string(index=False))

In [ ]:
# Read three disagreements: what the participant said, and each model's code and quote.
for row in disagree.head(3).itertuples():
    print(f"=== {row.transcript_id} ({row.group})")
    print(f"  model 1: {row.attitude} / {row.main_concern}   quote: {row.quote!r}")
    print(f"  model 2: {row.attitude_2} / {row.main_concern_2}   quote: {row.quote_2!r}\n")
    print(row.answers[:2000], "...\n")

Discuss as a team, and write down one sentence for each:

- In the three disagreements you read, **whose code would you defend to a reviewer**: model 1, model 2, or neither? Is it a model error, or a case the codebook does not decide?
- Is agreement higher for `attitude` or for `main_concern`? Why might that be?
- Suppose kappa is high. Does that show the codes are *right*? What might two models from different families still have in common? Before publishing, which interviews would a human have to read, and how many?

## Step 6 - The substantive question

Now, and only now, the research question: do the groups differ?

Choose the variable you want to test. The next cell prints row proportions for both variables (the share of each group in each category), then repeats the table for your variable using only the interviews on which both models agree, and tests whether it depends on group. Does the group difference hold when you keep only the interviews both models agree on?

In [ ]:
from scipy.stats import chi2_contingency

# TODO: which variable do you want to test for a group difference? "attitude" or "main_concern".
VARIABLE = ...
require(VARIABLE=VARIABLE)

print("ATTITUDE by group (row proportions)")
print(pd.crosstab(coded["group"], coded["attitude"], normalize="index").round(2), "\n")

print("MAIN CONCERN by group (row proportions)")
print(pd.crosstab(coded["group"], coded["main_concern"], normalize="index").round(2), "\n")

both_agree = coded[coded[VARIABLE] == coded[f"{VARIABLE}_2"]]
print(f"{VARIABLE.upper()} by group, only the {len(both_agree)} interviews on which both models agree")
print(pd.crosstab(both_agree["group"], both_agree[VARIABLE], normalize="index").round(2), "\n")

counts = pd.crosstab(coded["group"], coded[VARIABLE])
chi2, p, dof, expected = chi2_contingency(counts)
print(f"Does {VARIABLE} depend on group? chi2 = {chi2:.2f}, df = {dof}, p = {p:.3f}")
print(f"Cells with expected count below 5: {(expected < 5).sum()} of {expected.size} (the test is unreliable if there are many)")

Before you write anything down, be clear about what the data are:

- **Who are the participants?** People recruited to talk to Claude about how they use AI. They are not a sample of workers, creatives or scientists, and they may be more positive than their professions as a whole. Your finding can be about differences *between these groups of participants* at best.
- **The interviewer is an AI.** Would people tell an AI interviewer from an AI company that they dislike AI as openly as they would tell a human?
- **20 per group.** A difference of two or three interviews changes a proportion by 10-15 percentage points.

## Step 7 - Prepare your 3-minute report

Complete the methods paragraph below, as you would write it for a paper. Then prepare one finding (Step 6) and one validity threat.

*Methods (fill in):*

> We coded the answers of ___ interview participants (___ per group) from the Anthropic Interviewer dataset for attitude towards AI (___ categories) and main concern (___ categories). Coding was done by ___ (model, date of use) with the codebook and prompt in Appendix ___. We changed the starting codebook as follows: ___. All interviews were coded independently by a second model from a different family, ___, with the same codebook and prompt; agreement was ___% (κ = ___) for attitude and ___% (κ = ___) for main concern. Disagreements occurred mainly in ___. ___ of ___ supporting quotes could not be found verbatim in the transcripts; we handled these by ___.

## What to take from this

- The codebook is yours, and so are its weaknesses. Most disagreements with the model trace back to a definition that does not decide the case.
- Ask for evidence along with every code. A verbatim quote makes the model's work checkable, and checking it catches invented evidence cheaply.
- Report agreement with an independent second coder, with a chance-corrected statistic, and read the disagreements rather than only counting them. A second model from another family is a quick and cheap second coder. But two models agreeing does not prove they are right, so before publishing, a human still codes a random sample.
- A sound coding procedure does not fix a sample. What these interviews can tell you depends on who agreed to be interviewed, and by whom.

**Data:** Handa, K. et al. (2025). *Introducing Anthropic Interviewer: What 1,250 professionals told us about working with AI.* https://anthropic.com/research/anthropic-interviewer (data CC-BY).

# Additional ideas

Use OpenAI